<a href="https://colab.research.google.com/github/seekff/learn-python/blob/main/%E7%94%9F%E6%88%90%E5%99%A8%E5%92%8C%E8%BF%AD%E4%BB%A3%E5%99%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**四类核心对象的关系**

**① 容器（container）**

如 list、tuple、dict、set，内部存储多个元素。

**② 可迭代对象（iterable）**

能被 iter() 转换为迭代器。

判断方式：

    iter(obj)

    # 或 isinstance(obj, Iterable)

**③ 迭代器（iterator）**

具有 __next__() 方法

每次调用返回下一个元素

无元素时抛出 StopIteration

**④ 生成器（generator）**

生成器 = 懒人版迭代器（lazy iterator）  

按需生成数据，不占用大量内存。

In [3]:
def is_iterable(obj):
  try:
      iter(obj)
      return True
  except TypeError:
      return False

params = [
    1234,
    '1234',
    [1,2,3,4],
    set([1,2,3,4]),
    {'a':1, 'b':2, 'c':3, 'd':4},
    (1,2,3,4)
]

for param in params:
  print(param, is_iterable(param))


1234 False
1234 True
[1, 2, 3, 4] True
{1, 2, 3, 4} True
{'a': 1, 'b': 2, 'c': 3, 'd': 4} True
(1, 2, 3, 4) True


**生成器 vs 列表推导式：性能对比**

示例对比创建 1 亿个元素：

列表推导式：占用数 GB 内存，耗时长

生成器表达式：几乎不占内存，速度更快

生成器只在 next() 时生成元素，因此非常节省资源。

In [5]:

import os

import psutil

#显示当前python程序占用的内存大小
def show_memory_info(hint):
  pid = os.getpid()
  p = psutil.Process(pid)

  info = p.memory_full_info()
  memory = info.uss / 1024. / 1024
  print('{} memory used: {} MB'.format(hint, memory))

'''
迭代器[i for i in range(100000000)]可以生成一亿个元素的列表，
每个元素都存储在内存中，占用巨大内存
'''
def test_iterator():
  show_memory_info('initing iterator')
  list_1 = [i for i in range(100000000)]
  show_memory_info('after iterator initiated')
  print(sum(list_1))
  show_memory_info('after sum called')

'''
生成器(i for i in range(100000000))，调用next()函数的时候才会生成下一个变量
'''
def test_generator():
  show_memory_info('initing generator')
  list_2 = (i for i in range(100000000))
  show_memory_info('after generator initiated')
  print(sum(list_2))
  show_memory_info('after sum called')

if __name__ == '__main__':
  test_iterator()
  test_generator()



initing iterator memory used: 167.08203125 MB
after iterator initiated memory used: 3995.0390625 MB
4999999950000000
after sum called memory used: 3995.0390625 MB
initing generator memory used: 170.87890625 MB
after generator initiated memory used: 170.87890625 MB
4999999950000000
after sum called memory used: 170.87890625 MB


**自定义生成器（yield 的魔法）**

示例：


    def generator(k):
      i = 1
      while True:
          yield i ** k
          i += 1
特点：

yield 会暂停函数执行

下次 next() 从暂停处继续

局部变量不会丢失

可以构造 无限序列

In [10]:
'''数学恒等式：
(1 + 2 + 3 + ... + n)^2 = 1^3 + 2^3 + 3^3 + ... + n^3
'''
def generator(k):
  i = 1
  while True:
      yield i ** k
      i += 1

gen_1 = generator(1)
gen_3 = generator(3)

print(gen_1)
print(gen_3)

def get_sum(n):
  sum_1,sum_3 = 0,0
  for i in range(n):
    next_1 = next(gen_1)
    next_3 = next(gen_3)
    print('netx_1={}, next_3={}'.format(next_1, next_3))
    sum_1 += next_1
    sum_3 += next_3
  print(sum_1*sum_1, sum_3)

get_sum(8)

<generator object generator at 0x782422492680>
<generator object generator at 0x782410efb100>
netx_1=1, next_3=1
netx_1=2, next_3=8
netx_1=3, next_3=27
netx_1=4, next_3=64
netx_1=5, next_3=125
netx_1=6, next_3=216
netx_1=7, next_3=343
netx_1=8, next_3=512
1296 1296


In [13]:

def index_normal(L, target):
  result = []
  for i, num in enumerate(L):
    if num == target:
      result.append(i)
  return result

'''
返回的是Generator对象，需要使用list转换为列表后才能打印
'''
def index_generator(L, target):
  for i, num in enumerate(L):
    if num == target:
      yield i

L = [1,6,2,4,5,2,8,6,3,2]
target = 2

print(index_normal(L, target))
print(list(index_generator(L, target)))



[2, 5, 9]
[2, 5, 9]


In [17]:

'''
i in b for i in a
等同于
gen = (i for i in a)  #产生一个生成器

i in b
等同于
while True:
  val = next(b)
  if val == i:
    yield True

all() 函数：用来判断一个迭代器的元素是否全部是True,如果是则返回True，否则返回False

'''
def is_subsequence(a, b):
  b = iter(b) #转换成一个迭代器
  return all(i in b for i in a)

b = [1,2,3,4,5]
a = [1,3,5]
print(is_subsequence(a, b))
a = [1,4,3]
print(is_subsequence(a, b))

print('----------------------------------')

s = (i for i in range(5))
print(2 in s)
print(4 in s)
print(3 in s)


True
False
----------------------------------
True
True
False


**生成器的限制与注意事项**

生成器只能遍历一次

遍历完后再次 next() → StopIteration

若要重新遍历，必须重新创建生成器对象

使用 in 会消耗迭代器的元素（指针前移）